In [1]:
from typing_extensions import Annotated, TypedDict
from functools import partial
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver  
from langgraph.graph.message import add_messages
from langchain.tools import tool
from database.manager import DatabaseManager
from datetime import datetime
from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages



In [2]:
from graph.nodes import *

def router_after_intent(state: State):
    if state.get("next_action") == "register_expense":
        return "execute_add_registry_node"
    return END  # END é uma constante especial, funciona sem mapeamento também


def build_graph(agent, db_manager):

    graph = StateGraph(State)
    graph.add_node("detect_intent", partial(detect_intent_node, agent=agent))
    graph.add_node("execute_add_registry_node", partial(execute_ad_registry_node, db_manager=db_manager))

    graph.add_edge(START, "detect_intent")
    graph.add_conditional_edges("detect_intent", router_after_intent)
    graph.add_edge("execute_add_registry_node", END)

    graph = graph.compile()
    return graph

